# 03 Transformation Test

This notebook is used to prototype the transformation logic before converting it into PostgreSQL stored procedures.

For each table, the structure is:

```text
1. Query the raw table
2. Apply the transformation SELECT statement

```

In order to compare the raw output with the transformed output


## 1. Connect to PostgreSQL

This notebook reuses the database connection from `src/db_connection.py`.

In [1]:
from pathlib import Path
import sys
import pandas as pd

BASE_DIR = Path.cwd().parent
SRC_DIR = BASE_DIR / "src"

sys.path.append(str(SRC_DIR))

from db_connection import get_engine

engine = get_engine()

2026-08-25 18:28:58,302 | INFO | db_connection | Database environment variables validated successfully.
2026-08-25 18:28:58,303 | INFO | db_connection | Building database URL for host=localhost, port=5432, database=hospital_management, user=postgres
2026-08-25 18:28:58,304 | INFO | db_connection | Creating SQLAlchemy engine.


## 2. Helper function

This helper runs a SQL query in PostgreSQL and returns the result as a pandas DataFrame.


In [2]:
def run_query(query : str) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql_query(query, conn)

# 3. Patients transformation

Business rules:
1. Standardize `gender` to `F`, `M`, or `O`.
2. Convert `date_of_birth` to `DATE`.
3. Keep only numbers in `contact_number`.
4. Convert `registration_date` to `DATE`.
5. Convert `email` to lowercase.


## 3.1 Patients — raw data

First, retrieve the data as it currently exists in the `raw.patients` table.


In [4]:
raw_patients_query = """
SELECT
    patient_id,
    first_name,
    last_name,
    gender,
    date_of_birth,
    contact_number,
    address,
    registration_date,
    insurance_provider,
    insurance_number,
    email,
    source_file
FROM raw.patients
WHERE patient_id IS NOT NULL
ORDER BY patient_id;
"""

raw_patients_df = run_query(raw_patients_query)
raw_patients_df.head(3)

,patient_id,first_name,last_name,gender,date_of_birth,contact_number,address,registration_date,insurance_provider,insurance_number,email,source_file
0,P001,David,Williams,Female,04/06/1955,(693)9585183,789 Pine Rd,23/06/2022,WellnessCorp,INS840674,david.williams@mail.com,patients00.csv
1,P002,Emily,Smith,Female,12/10/1984,8228188767,321 Maple Dr,15/01/2022,PulseSecure,INS354079,emily.smith@mail.com,patients00.csv
2,P003,Laura,Jones,MALE,21/08/1977,(839)7029847,321 Maple Dr,07/02/2022,PulseSecure,INS650929,LAURA.JONES@MAIL.COM,patients00.csv


## 3.2 Patients — transformed data

Now apply the transformation logic using a `SELECT` statement.


In [5]:
transformed_patients_query = """
SELECT
    patient_id,
    first_name,
    last_name,
    CASE
        WHEN LOWER(TRIM(gender)) = 'female' THEN 'F'
        WHEN LOWER(TRIM(gender)) = 'male' THEN 'M'
        WHEN LOWER(TRIM(gender)) = 'f' THEN 'F'
        WHEN LOWER(TRIM(gender)) = 'm' THEN 'M'
        ELSE 'O'
    END AS gender,
    TO_DATE(date_of_birth, 'DD/MM/YYYY') AS date_of_birth,
    REGEXP_REPLACE(contact_number, '[^0-9]', '', 'g') AS contact_number,
    address,
    TO_DATE(registration_date, 'DD/MM/YYYY') AS registration_date,
    insurance_provider,
    insurance_number,
    LOWER(TRIM(email)) AS email,
    source_file
FROM raw.patients
WHERE patient_id IS NOT NULL
ORDER BY patient_id;
"""

patients_df = run_query(transformed_patients_query)
patients_df.head(3)

,patient_id,first_name,last_name,gender,date_of_birth,contact_number,address,registration_date,insurance_provider,insurance_number,email,source_file
0,P001,David,Williams,F,1955-06-04,6939585183,789 Pine Rd,2022-06-23,WellnessCorp,INS840674,david.williams@mail.com,patients00.csv
1,P002,Emily,Smith,F,1984-10-12,8228188767,321 Maple Dr,2022-01-15,PulseSecure,INS354079,emily.smith@mail.com,patients00.csv
2,P003,Laura,Jones,M,1977-08-21,8397029847,321 Maple Dr,2022-02-07,PulseSecure,INS650929,laura.jones@mail.com,patients00.csv


# 4. Doctors transformation

Business rules:
1. Format `specialization` using title case.
2. Keep only numbers in `phone_number`.
3. Convert `years_experience` to integer.
4. Convert `email` to lowercase.


## 4.1 Doctors — raw data

In [6]:
raw_doctors_query = """
SELECT
    doctor_id,
    first_name,
    last_name,
    specialization,
    phone_number,
    years_experience,
    hospital_branch,
    email,
    source_file
FROM raw.doctors
WHERE doctor_id IS NOT NULL
ORDER BY doctor_id;
"""

raw_doctors_df = run_query(raw_doctors_query)
raw_doctors_df.head(10)

,doctor_id,first_name,last_name,specialization,phone_number,years_experience,hospital_branch,email,source_file
0,D001,David,Taylor,DERMATOLOGY,8322010158,17,Westside Clinic,dr.david.taylor@hospital.com,doctors01.csv
1,D002,Jane,Davis,Pediatrics,9004382050,24,Eastside Clinic,DR.JANE.DAVIS@HOSPITAL.COM,doctors01.csv
2,D003,Jane,Smith,Pediatrics,(873)7740598,19,Eastside Clinic,dr.jane.smith@hospital.com,doctors01.csv
3,D004,David,Jones,PEDIATRICS,659-422-1991,28,Central Hospital,dr.david.jones@hospital.com,doctors01.csv
4,D005,Sarah,Taylor,dermatology,911-853-8547,26,Central Hospital,DR.SARAH.TAYLOR@HOSPITAL.COM,doctors01.csv
5,D006,Alex,Davis,Pediatrics,6570137231,23,Central Hospital,DR.ALEX.DAVIS@HOSPITAL.COM,doctors01.csv
6,D007,Robert,Davis,Oncology,8217493115,26,Westside Clinic,dr.robert.davis@hospital.com,doctors01.csv
7,D008,Linda,Brown,Dermatology,9069162601,5,Westside Clinic,DR.LINDA.BROWN@HOSPITAL.COM,doctors01.csv
8,D009,Sarah,Smith,PEDIATRICS,(738)7087517,26,Central Hospital,DR.SARAH.SMITH@HOSPITAL.COM,doctors01.csv
9,D010,Linda,Wilson,Oncology,6176383634,21,Eastside Clinic,DR.LINDA.WILSON@HOSPITAL.COM,doctors01.csv


## 4.2 Doctors — transformed data

# 5. Appointments transformation

Business rules:
1. Convert `appointment_date` to `DATE`.
2. Convert `appointment_time` to `TIME`.
3. Format `status` using title case.


## 5.1 Appointments — raw data

## 5.2 Appointments — transformed data

# 6. Treatments transformation

Business rules:
1. Convert `treatment_date` to `DATE`.


## 6.1 Treatments — raw data

## 6.2 Treatments — transformed data

# 7. Billing transformation

Business rules:
1. Convert `bill_date` to `DATE`.
2. Clean `amount` and convert it to `NUMERIC`.
3. Standardize `payment_method`.


## 7.1 Billing — raw data

## 7.2 Billing — transformed data